In [1]:
import cv2
import numpy as np
import os

def order_points(pts):
    """Сортирует точки: топ-лево, топ-право, низ-право, низ-лево"""
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

In [2]:
image_path = 'кетоны/свет1-вид под углом/full.jpg'  
img = cv2.imread(image_path)
if img is None:
    print("Ошибка: Файл не найден")
else:
    print("Файл загружен")
   

Файл загружен


In [3]:
# 1. Предобработка
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

In [4]:
blurred = cv2.GaussianBlur(gray, (7, 7), 0)

In [5]:
thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]

In [6]:
# 2. Поиск контуров
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

strip_cnt = max(contours, key=cv2.contourArea)

# 3. Аппроксимация контура до 4-х точек (углов трапеции)
peri = cv2.arcLength(strip_cnt, True)
approx = cv2.approxPolyDP(strip_cnt, 0.02 * peri, True)

if len(approx) != 4:
    # Если не нашли 4 четких угла, используем ограничивающий прямоугольник
    rect = cv2.minAreaRect(strip_cnt)
    box = cv2.boxPoints(rect)
else:
    box = approx.reshape(4, 2)

# Исправляем ошибку np.int0
box = box.astype(int)

# 4. Коррекция перспективы (выпрямление)
rect_pts = order_points(box.astype("float32"))
(tl, tr, br, bl) = rect_pts

# Считаем ширину и высоту новой картинки
width_a = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
width_b = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
max_width = max(int(width_a), int(width_b))

height_a = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
height_b = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
max_height = max(int(height_a), int(height_b))

dst = np.array([
    [0, 0],
    [max_width - 1, 0],
    [max_width - 1, max_height - 1],
    [0, max_height - 1]], dtype="float32")

M = cv2.getPerspectiveTransform(rect_pts, dst)
warped = cv2.warpPerspective(img, M, (max_width, max_height))

# Если полоска встала вертикально — поворачиваем горизонтально
if max_height > max_width:
    warped = cv2.rotate(warped, cv2.ROTATE_90_CLOCKWISE)

